# Bloque 4: Validación Robusta
## Validación correcta: detectar leakage, elegir métricas, evaluar OOD

**Objetivo**: Entender cómo validar modelos correctamente en producción.
- Qué CV variante usar
- Detectar leakage (el asesino silencioso)
- Elegir métrica apropiada
- Evaluar Out-of-Distribution (OOD)
- Framework defensivo integral

## Setup

In [ ]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_classification
from sklearn.model_selection import (
    train_test_split, cross_val_score, cross_validate, KFold, 
    StratifiedKFold, TimeSeriesSplit
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_curve, precision_recall_curve, average_precision_score
)
from xgboost import XGBClassifier
from scipy.stats import ks_2samp
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
np.random.seed(42)

print("✓ Setup listo")

## Dataset

In [ ]:
# Dataset
np.random.seed(42)
X, y = make_classification(
    n_samples=2000,
    n_features=20,
    n_informative=12,
    n_redundant=5,
    n_classes=2,
    weights=[0.7, 0.3],  # Desbalance
    random_state=42
)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=42, stratify=y_temp
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Desbalance: Clase 0={np.sum(y==0)}, Clase 1={np.sum(y==1)}")

---
# SECCIÓN 1: Cross-Validation - Variantes y cuándo importa

## El problema con Hold-out simple

Train/Val/Test = fácil pero usa solo ~80% de datos para entrenar.

**Cross-Validation** = usa todos los datos para entrenar + validar sin fuga.

In [ ]:
print("SECCIÓN 1: Cross-Validation")
print("="*70)

# Comparar: Hold-out vs CV
model = XGBClassifier(n_estimators=50, max_depth=5, random_state=42)

# Hold-out simple
model.fit(X_train, y_train)
holdout_auc = roc_auc_score(y_val, model.predict_proba(X_val)[:, 1])

# 5-Fold CV
cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
cv_mean = cv_scores.mean()
cv_std = cv_scores.std()

print(f"\n1. HOLD-OUT (Train/Val simple):")
print(f"   Val AUC: {holdout_auc:.4f}")
print(f"   → Usa solo ~80% datos para train, 20% para val")

print(f"\n2. 5-FOLD CV:")
print(f"   CV AUCs: {[f'{s:.4f}' for s in cv_scores]}")
print(f"   Mean: {cv_mean:.4f} ± {cv_std:.4f}")
print(f"   → Usa 100% datos, muestreo sin fuga, mejor estimación")

## Las variantes de CV

In [ ]:
# VARIANTES DE CV
print("\n" + "="*70)
print("VARIANTES DE CROSS-VALIDATION")
print("="*70)

# 1. K-Fold simple
kfold = KFold(n_splits=5, shuffle=True, random_state=42)
kfold_scores = cross_val_score(model, X_train, y_train, cv=kfold, scoring='roc_auc')

# 2. Stratified K-Fold (respeta desbalance)
stratified = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
stratified_scores = cross_val_score(model, X_train, y_train, cv=stratified, scoring='roc_auc')

# 3. Time Series CV (para datos temporales)
# Simulamos: X_train tiene orden temporal
tscv = TimeSeriesSplit(n_splits=5)
tscv_scores = cross_val_score(model, X_train, y_train, cv=tscv, scoring='roc_auc')

print(f"\n1. K-Fold simple:")
print(f"   Scores: {[f'{s:.4f}' for s in kfold_scores]}")
print(f"   Mean: {kfold_scores.mean():.4f} ± {kfold_scores.std():.4f}")
print(f"   → Splits aleatorios, riesgo de desbalance por fold")

print(f"\n2. Stratified K-Fold:")
print(f"   Scores: {[f'{s:.4f}' for s in stratified_scores]}")
print(f"   Mean: {stratified_scores.mean():.4f} ± {stratified_scores.std():.4f}")
print(f"   → ✓ Respeta proporción clase en cada fold")
print(f"   → Varianza más baja (splits balanceados)")

print(f"\n3. Time Series CV:")
print(f"   Scores: {[f'{s:.4f}' for s in tscv_scores]}")
print(f"   Mean: {tscv_scores.mean():.4f} ± {tscv_scores.std():.4f}")
print(f"   → ✓ Respeta orden temporal (sin información del futuro)")
print(f"   → Para datos con dependencia temporal")

## Tabla de decisión: Cuál CV usar

In [ ]:
decision_cv = pd.DataFrame({
    'Tu datos': [
        'Balanceados, no temporales',
        'Desbalanceados',
        'Series temporal / dependencia',
        'Prototipo rápido (sin tiempo)'
    ],
    'Usa': [
        'K-Fold (5-10)',
        'Stratified K-Fold',
        'Time Series CV',
        'Hold-out 80/20'
    ],
    'Razón': [
        'Simple, rápido',
        'Mantiene proporción clase → varianza baja',
        'No filtra información del futuro',
        'Rápido, pero menos robusto'
    ]
})

print("\nTABLA: Qué CV elegir\n")
print(decision_cv.to_string(index=False))

### ✅ CONCLUSIÓN Sección 1

**K-Fold es lo estándar**, Stratified si hay desbalance, Time Series si hay dependencia temporal.

**CV da estimación mejor** que hold-out simple (usa 100% datos, más confiable).

**Regla**: Si desbalance presente → SIEMPRE Stratified K-Fold (no K-Fold).

---
# SECCIÓN 2: Data Leakage - El asesino silencioso

## ¿Qué es leakage?

**Información del futuro (o del test set) entra en train** → AUC falsamente alta.

Es invisible pero destruye el modelo en producción.

## Gotcha #1: Leakage temporal

In [ ]:
print("\nSECCIÓN 2: DATA LEAKAGE")
print("="*70)
print("\nGOTCHA #1: Leakage Temporal\n")

# Simular: datos con 'fecha'
# Objetivo: predecir evento Q (mañana)
# LEAK: usar feature calculado HOY (que depende del futuro)

# Datos "correctos"
np.random.seed(42)
n_samples = 500
dates = pd.date_range('2024-01-01', periods=n_samples, freq='D')

# Feature legítimo: temperatura de AYER (pasado)
temp_yesterday = np.random.normal(20, 5, n_samples)

# Feature CON LEAK: promedio temperatura de HOY (info del futuro)
# (Simulamos que en train sí tenemos esta info, pero en prod no)
temp_today = temp_yesterday + np.random.normal(2, 1, n_samples)

# Target: hubo evento (lluvia) mañana
y_leak = (temp_today > 22).astype(int) + np.random.binomial(1, 0.1, n_samples)

# Split train/test (INCORRECTO: random split, futura info entra en train)
X_data = np.column_stack([temp_yesterday, temp_today])
X_train_leak, X_test_leak, y_train_leak, y_test_leak = train_test_split(
    X_data, y_leak, test_size=0.2, random_state=42
)

# Modelo CON leak
model_leak = XGBClassifier(n_estimators=50, max_depth=3, random_state=42)
model_leak.fit(X_train_leak, y_train_leak)
auc_leak = roc_auc_score(y_test_leak, model_leak.predict_proba(X_test_leak)[:, 1])

# Modelo CORRECTO: solo temp_yesterday
X_train_clean = X_train_leak[:, :1]
X_test_clean = X_test_leak[:, :1]
model_clean = XGBClassifier(n_estimators=50, max_depth=3, random_state=42)
model_clean.fit(X_train_clean, y_train_leak)
auc_clean = roc_auc_score(y_test_leak, model_clean.predict_proba(X_test_clean)[:, 1])

print(f"❌ CON LEAK (temp_today): AUC = {auc_leak:.4f}")
print(f"✅ SIN LEAK (temp_yesterday): AUC = {auc_clean:.4f}")
print(f"\n⚠️  Diferencia: {(auc_leak - auc_clean):.4f}")
print(f"\nEn producción: modelo con leak fallará (no tiene temp_today en el momento de predicción).")

## Gotcha #2: Leakage estadístico (transformaciones globales)

In [ ]:
print("\nGOTCHA #2: Leakage Estadístico\n")

# LEAK: normalizar ANTES de split train/test
# (Test set stats entran en normalización de train)

np.random.seed(42)
X_full = np.random.normal(0, 1, (1000, 5))
y_full = np.random.binomial(1, 0.5, 1000)

# ❌ INCORRECTO: normalizar TODO, luego split
scaler_wrong = StandardScaler()
X_normalized_wrong = scaler_wrong.fit_transform(X_full)  # Fit en TODOS (incluyendo test)
X_train_wrong, X_test_wrong, y_train_w, y_test_w = train_test_split(
    X_normalized_wrong, y_full, test_size=0.2, random_state=42
)

# ✅ CORRECTO: split PRIMERO, normalizar solo en train
X_train_correct, X_test_correct, y_train_c, y_test_c = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)
scaler_correct = StandardScaler()
X_train_correct = scaler_correct.fit_transform(X_train_correct)  # Fit solo en train
X_test_correct = scaler_correct.transform(X_test_correct)  # Transform test con stats de train

# Entrenar modelos
model_w = XGBClassifier(n_estimators=30, random_state=42)
model_c = XGBClassifier(n_estimators=30, random_state=42)

model_w.fit(X_train_wrong, y_train_w)
model_c.fit(X_train_correct, y_train_c)

auc_wrong = roc_auc_score(y_test_w, model_w.predict_proba(X_test_wrong)[:, 1])
auc_correct = roc_auc_score(y_test_c, model_c.predict_proba(X_test_correct)[:, 1])

print(f"❌ INCORRECTO (normalize antes de split): AUC = {auc_wrong:.4f}")
print(f"✅ CORRECTO (split, normalize en train): AUC = {auc_correct:.4f}")
print(f"\nDiferencia: {abs(auc_wrong - auc_correct):.4f}")
print(f"\n⚠️  El leak estadístico es sutil pero real.")

## Gotcha #3: Feature selection con target (peor leak)

In [ ]:
print("\nGOTCHA #3: Feature Selection CON Target\n")

# Crear datos con features irrelevantes
np.random.seed(42)
X_full = np.random.normal(0, 1, (1000, 20))  # 20 features
y_full = np.random.binomial(1, 0.5, 1000)   # target aleatorio

# ❌ INCORRECTO: seleccionar features basado en TODOS los datos
# (correlación con target es por azar, pero parecerá útil)
correlations = np.array([np.corrcoef(X_full[:, i], y_full)[0, 1] for i in range(20)])
top_features_wrong = np.argsort(np.abs(correlations))[-5:]  # Top 5 features (CON leak)

X_train_w, X_test_w, y_train_w, y_test_w = train_test_split(
    X_full[:, top_features_wrong], y_full, test_size=0.2, random_state=42
)

# ✅ CORRECTO: seleccionar features SOLO en train
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_full, y_full, test_size=0.2, random_state=42
)
correlations_train = np.array([np.corrcoef(X_train_c[:, i], y_train_c)[0, 1] for i in range(20)])
top_features_correct = np.argsort(np.abs(correlations_train))[-5:]

X_train_c = X_train_c[:, top_features_correct]
X_test_c = X_test_c[:, top_features_correct]

# Entrenar
model_w = XGBClassifier(n_estimators=30, random_state=42)
model_c = XGBClassifier(n_estimators=30, random_state=42)

model_w.fit(X_train_w, y_train_w)
model_c.fit(X_train_c, y_train_c)

auc_w = roc_auc_score(y_test_w, model_w.predict_proba(X_test_w)[:, 1])
auc_c = roc_auc_score(y_test_c, model_c.predict_proba(X_test_c)[:, 1])

print(f"❌ INCORRECTO (select features en full data): AUC = {auc_w:.4f}")
print(f"✅ CORRECTO (select features en train): AUC = {auc_c:.4f}")
print(f"\n⚠️  Feature selection EN TODO el dataset es el PEOR leak. Máxima vigilancia.")

## Checklist: Detectar leakage

In [ ]:
print("\nCHECKLIST: Detectar LEAKAGE\n")
print("="*70)

checklist_leak = pd.DataFrame({
    '🚩': ['❌', '❌', '❌', '❌', '❌'],
    'Señal de alerta': [
        'Train AUC >> Val AUC (diferencia > 10%)',
        'Feature muy correlacionada con target (ρ > 0.9)',
        'Feature que depende del futuro (temporalmente)',
        'Normalización/scaling ANTES de split',
        'Feature selection EN TODO el dataset'
    ],
    'Acción': [
        'Revisar validación, sospechar leak',
        'Verificar que feature no es derivado del target',
        'Asegurar feature está disponible en predicción',
        'SIEMPRE: split PRIMERO, fit en train',
        'Usar CV: fit transformación en train de cada fold'
    ]
})

print(checklist_leak.to_string(index=False))
print("\n" + "="*70)

### ✅ CONCLUSIÓN Sección 2

**Data leakage es invisible pero letal**: AUC high pero falla en producción.

**3 tipos**: temporal (futuro entra), estadístico (stats test en train), directo (feature selection).

**Regla de oro**: Split PRIMERO. Todo (normalización, selección) en train. Nunca toques test.

---
# SECCIÓN 3: Métricas apropiadas - Cuál elegir

## El problema: métrica equivocada = decisión mala

In [ ]:
print("\nSECCIÓN 3: MÉTRICAS APROPIADAS")
print("="*70)

# Entrenar modelo
model = XGBClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Predicciones
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calcular todas las métricas
auc = roc_auc_score(y_test, y_pred_proba)
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
ap = average_precision_score(y_test, y_pred_proba)

print(f"\nMétricas modelo test set:")
print(f"  AUC (ROC): {auc:.4f}")
print(f"  Accuracy: {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall: {rec:.4f}")
print(f"  F1-Score: {f1:.4f}")
print(f"  AP (Precision-Recall): {ap:.4f}")

## Cuándo usar cada métrica

In [ ]:
# Confusion matrix
tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

print(f"\nConfusion Matrix:")
print(f"  True Negatives (TN): {tn}")
print(f"  False Positives (FP): {fp}")
print(f"  False Negatives (FN): {fn}")
print(f"  True Positives (TP): {tp}")

print("\n" + "="*70)
print("TABLA: Métrica vs Contexto\n")

metrics_table = pd.DataFrame({
    'Métrica': ['AUC (ROC)', 'Accuracy', 'Precision', 'Recall (Sensitivity)', 'F1', 'AP (Avg Precision)'],
    'Fórmula': [
        'P(score_pos > score_neg)',
        '(TP+TN) / Total',
        'TP / (TP+FP)',
        'TP / (TP+FN)',
        '2*(Prec*Rec)/(Prec+Rec)',
        'Promedio Precisión en distintos thresholds'
    ],
    'Cuándo usar': [
        '✓ Clasificación general, datos desbalanceados',
        '✗ Evitar si desbalance (engaña)',
        '✓ Si costo FP alto (falsos alarmas caros)',
        '✓ Si costo FN alto (perder positivos es malo)',
        '✓ Balance Prec-Rec sin pesos',
        '✓ Mejor que AUC con desbalance extremo'
    ],
    'Riesgo': [
        'Puede ocultar mal rendimiento en clase minoritaria',
        'ALTO: con desbalance 70/30, predecir todo clase 0 = 70% accuracy',
        'Ignora FN (olvida positivos)',
        'Ignora FP (hace alarmas falsas)',
        'Simétrico (a veces no queremos)',
        'Más conservador que AUC'
    ]
})

print(metrics_table.to_string(index=False))

In [ ]:
# Simulación: modelo que predice TODO clase mayoritaria
print("\n" + "="*70)
print("DEMOSTRACIÓN: Accuracy engaña con desbalance\n")

# Modelo "predictor_todo_clase_0"
y_pred_naive = np.zeros(len(y_test))  # Siempre predice 0

acc_naive = accuracy_score(y_test, y_pred_naive)
prec_naive = precision_score(y_test, y_pred_naive, zero_division=0)
rec_naive = recall_score(y_test, y_pred_naive, zero_division=0)
f1_naive = f1_score(y_test, y_pred_naive, zero_division=0)

print(f"Modelo naive (predice todo clase 0):")
print(f"  Accuracy: {acc_naive:.4f} ⚠️  ¡ALTO!")
print(f"  Precision: {prec_naive:.4f}")
print(f"  Recall: {rec_naive:.4f} ❌ CERO (no detecta positivos)")
print(f"  F1: {f1_naive:.4f}")

print(f"\n⚠️  Accuracy = {acc_naive:.4f} parece bueno, pero es INÚTIL (recall=0).")
print(f"    F1 = {f1_naive:.4f} refleja la verdad: modelo es basura.")

In [ ]:
# Tabla de decisión
print("\n" + "="*70)
print("TABLA DE DECISIÓN: Qué métrica usar\n")

decision_metrics = pd.DataFrame({
    'Tu problema': [
        'Datos balanceados',
        'Datos desbalanceados (70/30)',
        'Costo FP > FN (falsos alarmas)',
        'Costo FN > FP (perder positivos)',
        'Desbalance extremo (95/5)'
    ],
    'Métrica primaria': [
        'Accuracy + AUC',
        'AUC + F1',
        'Precision',
        'Recall',
        'AP (Average Precision)'
    ],
    'Métrica secundaria': [
        'Precision, Recall',
        'Precision, Recall',
        'F1 para balance',
        'F1 para balance',
        'AUC (verificar)'
    ]
})

print(decision_metrics.to_string(index=False))

### ✅ CONCLUSIÓN Sección 3

**Con desbalance**: NUNCA Accuracy. Usa AUC + F1 + AP.

**Si costo FP alto** (falsos alarmas caros): Precision.

**Si costo FN alto** (perder positivos es malo): Recall.

**Regla**: AUC es seguro para datos desbalanceados. F1 es más conservador.

---
# SECCIÓN 4: Evaluación Out-of-Distribution (OOD) + Distribution Shift

## El problema: datos en producción NO son iguales a entrenamiento

In [ ]:
print("\nSECCIÓN 4: OUT-OF-DISTRIBUTION (OOD) + DISTRIBUTION SHIFT")
print("="*70)

# Crear test set con SHIFT de distribución
np.random.seed(123)
X_test_ood = X_test + np.random.normal(0.8, 0.5, X_test.shape)

# Entrenar modelo
model = XGBClassifier(n_estimators=100, max_depth=5, random_state=42)
model.fit(X_train, y_train)

# Evaluación
auc_clean = roc_auc_score(y_test, model.predict_proba(X_test)[:, 1])
auc_ood = roc_auc_score(y_test, model.predict_proba(X_test_ood)[:, 1])

print(f"\nTest set LIMPIO (misma distribución train): AUC = {auc_clean:.4f}")
print(f"Test set OOD (distribución diferente):      AUC = {auc_ood:.4f}")
print(f"\nCaída: {(auc_clean - auc_ood)*100:.2f}%")
print(f"\n⚠️  Distribution shift es REAL en producción.")

## Detectar OOD: KS-Test + Wasserstein

In [ ]:
# KS Test para detectar shift
print("\nDetectando OOD con KS-Test:\n")

feature_idx = 0
ks_stat, p_value = ks_2samp(X_train[:, feature_idx], X_test_ood[:, feature_idx])

print(f"Feature {feature_idx}:")
print(f"  KS-Statistic: {ks_stat:.4f}")
print(f"  P-value: {p_value:.6f}")
print(f"\n  Interpretación:")
if p_value < 0.05:
    print(f"    ✓ Distribuciones son DIFERENTES (p < 0.05)")
else:
    print(f"    ✗ Distribuciones parecen iguales (p >= 0.05)")

# Comparar múltiples features
print(f"\n" + "="*70)
print("KS-Test en TODOS los features:\n")

ks_results = []
for i in range(X_train.shape[1]):
    ks_stat, p_val = ks_2samp(X_train[:, i], X_test_ood[:, i])
    ks_results.append({'Feature': i, 'KS-Stat': ks_stat, 'P-Value': p_val})

ks_df = pd.DataFrame(ks_results)
ks_df['Shift?'] = ks_df['P-Value'] < 0.05
print(ks_df.head(10).to_string(index=False))

n_shifted = (ks_df['P-Value'] < 0.05).sum()
print(f"\n→ {n_shifted}/{len(ks_df)} features tienen distribución diferente")
print(f"→ Alerta: distribution shift detectado")

## Validación temporal (si datos son series)

In [ ]:
# Simulación: datos con drift temporal
print("\n" + "="*70)
print("Validación temporal: Test en diferentes períodos\n")

# Simular 12 meses de datos
np.random.seed(42)
n_samples_month = 50
n_months = 12

months = []
aucs_by_month = []

for month in range(1, n_months+1):
    # Generar datos del mes (con drift)
    X_month = np.random.normal(month*0.3, 1, (n_samples_month, 5))  # Drift en media
    y_month = np.random.binomial(1, 0.3 + month*0.02, n_samples_month)  # Drift en prevalencia
    
    # Evaluación
    preds = model.predict_proba(X_month)[:, 1]
    auc_month = roc_auc_score(y_month, preds) if len(np.unique(y_month)) > 1 else 0.5
    
    months.append(month)
    aucs_by_month.append(auc_month)

temporal_df = pd.DataFrame({
    'Mes': months,
    'AUC': aucs_by_month
})

print(temporal_df.to_string(index=False))

print(f"\n→ AUC degrada con el tiempo (mes 1: {aucs_by_month[0]:.4f} → mes 12: {aucs_by_month[-1]:.4f})")
print(f"→ Evidencia de distribution shift temporal")

In [ ]:
# Gráfica: AUC por mes (monitoreo)
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(months, aucs_by_month, 'o-', linewidth=2, markersize=8, label='AUC por mes')
ax.axhline(auc_clean, color='green', linestyle='--', linewidth=2, label=f'Baseline (test clean: {auc_clean:.3f})')
ax.fill_between(months, auc_clean - 0.05, auc_clean + 0.05, alpha=0.2, color='green', label='Rango aceptable')

ax.set_xlabel('Mes')
ax.set_ylabel('AUC')
ax.set_title('Monitoreo: AUC degradación por OOD/Distribution Shift')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim([0.4, 0.8])

plt.tight_layout()
plt.show()

print("\n→ Si AUC cae fuera del rango verde: reentrenar modelo")

### ✅ CONCLUSIÓN Sección 4

**OOD y Distribution Shift son inevitables** en producción.

**Detectar con**: KS-Test, Wasserstein, monitoreo temporal.

**Acción**: si AUC cae > 5% → alerta, posible reentrenamiento.

---
# SECCIÓN 5: Framework defensivo integral

In [ ]:
print("\nSECCIÓN 5: FRAMEWORK DEFENSIVO INTEGRAL")
print("="*70)

framework = """
PIPELINE DE VALIDACIÓN ROBUSTA EN PRODUCCIÓN:

1. PREPARACIÓN
   ├─ Split PRIMERO: train/val/test (60/20/20)
   ├─ Si temporal: train=pasado, test=futuro
   └─ Si desbalance: usar Stratified splits

2. PREVENCIÓN DE LEAKAGE
   ├─ Normalización/Scaling: fit solo en TRAIN
   ├─ Feature selection: fit solo en TRAIN
   ├─ Feature engineering: usar SOLO info disponible en predicción
   └─ Temporal: nunca usar info del futuro

3. CROSS-VALIDATION
   ├─ Si desbalance → Stratified K-Fold
   ├─ Si temporal → Time Series CV
   └─ CV de tuning: asegurar no tocar test

4. MÉTRICA CORRECTA
   ├─ Con desbalance: AUC (nunca Accuracy)
   ├─ Si costo FP alto: Precision
   ├─ Si costo FN alto: Recall
   └─ Siempre F1 para balance

5. EVALUACIÓN ROBUSTA
   ├─ Val AUC ≈ Test AUC (diferencia < 5%)
   ├─ Test AUC en múltiples períodos (temporal)
   ├─ Detectar OOD: KS-Test en features
   └─ Confusion matrix: revisar FP y FN

6. MONITOREO EN PRODUCCIÓN
   ├─ Tracking AUC en rolling windows
   ├─ Alertar si AUC cae > 5% del baseline
   ├─ KS-Test periódico en features
   └─ Reentrenar si degradación > umbral
"""

print(framework)

In [ ]:
# CHECKLIST FINAL
print("\n" + "="*70)
print("CHECKLIST FINAL: ¿Tu validación es robusta?\n")

final_checklist = pd.DataFrame({
    '✓': ['☑', '☑', '☑', '☑', '☑', '☑', '☑', '☑', '☑', '☑'],
    'Punto': [
        'Test set NUNCA se toca durante tuning/feature selection',
        'Normalización/Scaling: fit solo en TRAIN',
        'Usando CV apropiada (Stratified si desbalance)',
        'Métrica: AUC (no Accuracy) si desbalance',
        'Val AUC - Test AUC < 5% (no overfitting)',
        'Evaluado en múltiples períodos (si temporal)',
        'Detecté OOD con KS-Test',
        'Confusion matrix revisada (FP, FN)',
        'Plan de monitoreo en producción',
        'Documenté umbral de reentrenamiento'
    ]
})

print(final_checklist.to_string(index=False))
print("\n" + "="*70)
print("\n⚠️  Si NO checkeaste todo → hay riesgo de fracaso en producción.")

### ✅ CONCLUSIÓN Sección 5

**Validación robusta = prevención de leakage + CV correcta + métrica apropiada + OOD awareness + monitoreo.**

**No es científico sin esto.**

---
# RESUMEN FINAL

## Bloque 4: Validación Robusta

| Sección | Aprendiste | Impacto crítico |
|---------|-----------|------------------|
| 1 | CV variantes (K-Fold, Stratified, TS-CV) | Estimación confiable vs. hold-out sesgado |
| 2 | Leakage (temporal, estadístico, directo) | Detectar AUC falsamente altas |
| 3 | Métricas (AUC, Accuracy, Precision, Recall) | Decisión correcta según costo de errores |
| 4 | OOD + Distribution Shift | Prepararse para la realidad de producción |
| 5 | Framework defensivo integral | Pipeline completo y robusto |

## Lecciones finales

1. **Leakage es invisible** → La paranoia es buena. Revisa 10 veces.
2. **Accuracy es una trampa** → Con desbalance, AUC siempre.
3. **OOD es garantizado** → Monitoreá en producción, no confíes solo en test.
4. **CV correcta = confianza** → Stratified si desbalance, Time Series si temporal.
5. **Test set es sagrado** → Tócalo una sola vez, al final.

---

**Próximo paso**: Integra este framework en TU tuning del Bloque 3. Valida correctamente.